# Latin Script Analysis

This notebook examines whether the tokenizer bias observed for Indic scripts also appears in Latin-script languages — specifically German (EN→DE) and Spanish (EN→ES). Because both languages use the Latin alphabet, XLM-R tokenises them efficiently, but morphological richness (German compounding) and orthographic regularity (Spanish) create different fragmentation patterns.

Seven complementary metrics are computed from the raw text and XLM-R token counts alone — no GPU required.

**Inputs** (place in `../../results/extra_metrics/`):
- `tp_ip_ende_clean.csv`
- `tp_ip_enes_clean.csv`

**Outputs** (written to `../../results/extra_metrics/`):
- `latin_word_length_distribution.csv`
- `latin_word_token_correlation.csv`
- `latin_single_token_coverage.csv`
- `latin_mattr.csv`
- `latin_byte_premium.csv`
- `latin_compound_proxy.csv`
- `latin_MASTER_SUMMARY.csv`

## Setup

Install dependencies if needed:

```
pip install transformers pandas scipy openpyxl
```

The tokenizer used throughout is `xlm-roberta-base` — the same backbone used inside COMET v2.2.6 (Rei et al., 2022).

In [ ]:
import os
import re
import glob
import math
from collections import Counter

import pandas as pd
import numpy as np
from scipy.stats import spearmanr
from transformers import XLMRobertaTokenizerFast

RESULTS_DIR = "../../results/extra_metrics"
os.makedirs(RESULTS_DIR, exist_ok=True)

tokenizer = XLMRobertaTokenizerFast.from_pretrained("xlm-roberta-base")
print("Tokenizer loaded: xlm-roberta-base")

LANGS = {
    "de": ("tp_ip_ende_clean.csv", "German",  "hyp", "ref"),
    "es": ("tp_ip_enes_clean.csv", "Spanish", "hyp", "ref"),
}

dfs = {}
for lang, (fname, label, hyp_col, ref_col) in LANGS.items():
    path = os.path.join(RESULTS_DIR, fname)
    if not os.path.exists(path):
        print(f"  WARNING: {fname} not found — skipping {label}")
        continue
    dfs[lang] = {"df": pd.read_csv(path), "label": label, "hyp": hyp_col, "ref": ref_col}
    print(f"  Loaded {label}: {len(dfs[lang]['df'])} rows")

## Metric 1 — Word-Length Distribution

Long words tokenise into more subword pieces. We measure mean character length, median, and the proportion of words exceeding 10 and 15 characters. German is expected to score high here due to compound nouns; Spanish should be closer to the English baseline.

Foroutan et al. (2025) show that byte length is a strong predictor of token fragmentation across scripts.

In [ ]:
def word_lengths(series):
    words = " ".join(series.dropna().astype(str)).split()
    lengths = [len(w) for w in words if w]
    if not lengths:
        return {"mean_chars": float("nan"), "median_chars": float("nan"),
                "pct_gt10": float("nan"), "pct_gt15": float("nan")}
    return {
        "mean_chars":   round(np.mean(lengths), 3),
        "median_chars": round(np.median(lengths), 3),
        "pct_gt10":     round(100 * sum(l > 10 for l in lengths) / len(lengths), 2),
        "pct_gt15":     round(100 * sum(l > 15 for l in lengths) / len(lengths), 2),
    }

wl_rows = []
for lang, info in dfs.items():
    df = info["df"]
    for col_key in [info["hyp"], info["ref"]]:
        if col_key not in df.columns:
            continue
        stats = word_lengths(df[col_key])
        wl_rows.append({"Language": info["label"], "Column": col_key, **stats})

wl_df = pd.DataFrame(wl_rows)
print(wl_df.to_string(index=False))
wl_df.to_csv(os.path.join(RESULTS_DIR, "latin_word_length_distribution.csv"), index=False)
print(f"\nSaved: latin_word_length_distribution.csv")

## Metric 2 — Word-Length to Token-Count Correlation

If the tokenizer fragments words proportionally to their character length, Spearman ρ between character count and XLM-R token count should be high. This is a mechanistic test: it confirms that fragmentation is driven by surface-level word structure rather than semantic content.

We compute this correlation per language and column.

In [ ]:
def token_count(text):
    if not text or (isinstance(text, float) and math.isnan(text)):
        return 0
    return len(tokenizer.tokenize(str(text)))

def word_token_corr(series):
    words = " ".join(series.dropna().astype(str)).split()
    chars = [len(w) for w in words if w]
    toks  = [token_count(w) for w in words if w]
    if len(chars) < 10:
        return float("nan"), float("nan")
    rho, pval = spearmanr(chars, toks)
    return round(rho, 4), round(pval, 6)

corr_rows = []
for lang, info in dfs.items():
    df = info["df"]
    for col_key in [info["hyp"], info["ref"]]:
        if col_key not in df.columns:
            continue
        rho, pval = word_token_corr(df[col_key])
        corr_rows.append({"Language": info["label"], "Column": col_key,
                          "Spearman_rho": rho, "p_value": pval})

corr_df = pd.DataFrame(corr_rows)
print(corr_df.to_string(index=False))
corr_df.to_csv(os.path.join(RESULTS_DIR, "latin_word_token_correlation.csv"), index=False)
print(f"\nSaved: latin_word_token_correlation.csv")

## Metric 3 — Single-Token Vocabulary Coverage

A word type encoded as a single XLM-R token is one the tokenizer knows as a unit. We count the percentage of unique word types in each text that require only one token. Higher coverage means the vocabulary aligns well with the language; lower coverage implies more fragmentation.

This is the vocabulary-level counterpart to the sentence-level TP ratio computed in notebook 04.

In [ ]:
def single_token_coverage(series):
    words = set(" ".join(series.dropna().astype(str)).split())
    if not words:
        return float("nan")
    single = sum(1 for w in words if len(tokenizer.tokenize(w)) == 1)
    return round(100 * single / len(words), 2)

cov_rows = []
for lang, info in dfs.items():
    df = info["df"]
    for col_key in [info["hyp"], info["ref"]]:
        if col_key not in df.columns:
            continue
        pct = single_token_coverage(df[col_key])
        cov_rows.append({"Language": info["label"], "Column": col_key, "single_token_pct": pct})

cov_df = pd.DataFrame(cov_rows)
print(cov_df.to_string(index=False))
cov_df.to_csv(os.path.join(RESULTS_DIR, "latin_single_token_coverage.csv"), index=False)
print(f"\nSaved: latin_single_token_coverage.csv")

## Metric 4 — Morphological Complexity (MATTR)

Moving-Average Type-Token Ratio (MATTR) measures lexical diversity using a sliding window of 50 words, making the score length-independent. A higher MATTR indicates greater morphological richness — more distinct word forms — which in inflected or agglutinative languages often corresponds to higher tokenizer fragmentation.

MATTR is used here as a proxy for morphological complexity, linking tokenizer behaviour to linguistic structure (Covington & McFall, 2010).

In [ ]:
def mattr(series, window=50):
    words = " ".join(series.dropna().astype(str)).lower().split()
    words = [re.sub(r"[^\w]", "", w) for w in words if w]
    if len(words) < window:
        return float("nan")
    ttrs = []
    for i in range(len(words) - window + 1):
        win = words[i : i + window]
        ttrs.append(len(set(win)) / window)
    return round(np.mean(ttrs), 4)

mattr_rows = []
for lang, info in dfs.items():
    df = info["df"]
    for col_key in [info["hyp"], info["ref"]]:
        if col_key not in df.columns:
            continue
        val = mattr(df[col_key])
        mattr_rows.append({"Language": info["label"], "Column": col_key, "MATTR": val})

mattr_df = pd.DataFrame(mattr_rows)
print(mattr_df.to_string(index=False))
mattr_df.to_csv(os.path.join(RESULTS_DIR, "latin_mattr.csv"), index=False)
print(f"\nSaved: latin_mattr.csv")

## Metric 5 — Byte Premium

The byte premium measures how many more UTF-8 bytes the target language text uses compared to the English source. For Latin-script languages this is close to 1.0 (no multi-byte characters), so any deviation captures encoding overhead from diacritics and special characters.

The formula mirrors the TP ratio: `byte_premium = mean(bytes(target)) / mean(bytes(source))`. Values above 1.0 indicate the target sentences are byte-heavier than their English sources.

In [ ]:
def byte_premium(src_series, tgt_series):
    src_bytes = src_series.dropna().astype(str).apply(lambda x: len(x.encode("utf-8")))
    tgt_bytes = tgt_series.dropna().astype(str).apply(lambda x: len(x.encode("utf-8")))
    if src_bytes.mean() == 0:
        return float("nan")
    return round(tgt_bytes.mean() / src_bytes.mean(), 4)

bp_rows = []
for lang, info in dfs.items():
    df = info["df"]
    # try common source column names
    src_col = next((c for c in ["src", "source", "Source"] if c in df.columns), None)
    if src_col is None:
        print(f"  WARNING: no source column found for {info['label']} — skipping byte premium")
        continue
    for col_key in [info["hyp"], info["ref"]]:
        if col_key not in df.columns:
            continue
        val = byte_premium(df[src_col], df[col_key])
        bp_rows.append({"Language": info["label"], "Column": col_key, "byte_premium": val})

bp_df = pd.DataFrame(bp_rows) if bp_rows else pd.DataFrame(columns=["Language","Column","byte_premium"])
print(bp_df.to_string(index=False))
bp_df.to_csv(os.path.join(RESULTS_DIR, "latin_byte_premium.csv"), index=False)
print(f"\nSaved: latin_byte_premium.csv")

## Metric 6 — Compound-Word Proxy (German)

German compound nouns are a well-known source of tokenizer fragmentation: a single semantic unit may split into many subword pieces. We use a simple proxy: the percentage of word tokens that require three or more XLM-R subword tokens.

This metric is computed for both languages. Spanish, which lacks productive nominal compounding, serves as a natural baseline.

In [ ]:
def compound_proxy(series, threshold=3):
    words = " ".join(series.dropna().astype(str)).split()
    if not words:
        return float("nan")
    heavy = sum(1 for w in words if len(tokenizer.tokenize(w)) >= threshold)
    return round(100 * heavy / len(words), 2)

cp_rows = []
for lang, info in dfs.items():
    df = info["df"]
    for col_key in [info["hyp"], info["ref"]]:
        if col_key not in df.columns:
            continue
        val = compound_proxy(df[col_key])
        cp_rows.append({"Language": info["label"], "Column": col_key, "pct_3plus_tokens": val})

cp_df = pd.DataFrame(cp_rows)
print(cp_df.to_string(index=False))
cp_df.to_csv(os.path.join(RESULTS_DIR, "latin_compound_proxy.csv"), index=False)
print(f"\nSaved: latin_compound_proxy.csv")

## Master Summary

All six metric tables are joined into a single summary CSV. Each row represents one language–column combination with all metrics as columns, making it straightforward to compare German and Spanish across every dimension in a single view.

In [ ]:
frames = [wl_df, corr_df, cov_df, mattr_df, bp_df, cp_df]
summary = frames[0].copy()
for f in frames[1:]:
    if f.empty:
        continue
    merge_keys = [c for c in ["Language", "Column"] if c in f.columns]
    summary = summary.merge(f, on=merge_keys, how="outer")

summary.to_csv(os.path.join(RESULTS_DIR, "latin_MASTER_SUMMARY.csv"), index=False)
print(summary.to_string(index=False))
print(f"\nMaster summary saved: latin_MASTER_SUMMARY.csv")

## References

Cross-lingual tokenization fairness:
Foroutan, N., Meister, C., Paul, D., Niklaus, J., Ahmadi, S., Bosselut, A., & Sennrich, R. (2025). Parity-Aware Byte-Pair Encoding: Improving Cross-lingual Fairness in Tokenization. arXiv:2508.04796. https://arxiv.org/abs/2508.04796

Tokenization and representation biases:
Kanjirangat, V., Samardžić, T., Dolamic, L., & Rinaldi, F. (2025). Tokenization and Representation Biases in Multilingual Models on Dialectal NLP Tasks. EMNLP 2025, pp. 23992–24010. https://arxiv.org/abs/2509.20045

XLM-RoBERTa (tokenizer backbone):
Conneau, A., Khandelwal, K., Goyal, N., Chaudhary, V., Wenzek, G., Guzmán, F., Grave, E., Ott, M., Zettlemoyer, L., & Stoyanov, V. (2020). Unsupervised Cross-lingual Representation Learning at Scale. ACL 2020. https://arxiv.org/abs/1911.02116

COMET (wmt22-comet-da):
Rei, R., Stewart, C., Farinha, A. C., & Lavie, A. (2020). COMET: A Neural Framework for MT Evaluation. EMNLP 2020. https://aclanthology.org/2020.emnlp-main.213

MATTR (morphological complexity measure):
Covington, M. A., & McFall, J. D. (2010). Cutting the Gordian Knot: The Moving-Average Type–Token Ratio (MATTR). Journal of Quantitative Linguistics, 17(2), 94–100.